# Neuron ASD
### An open platform for exploring receptor agonist/inhibitor modulations toward a typically-developing (TD) profile from EEG in autism

This notebook lets you run **Neuron ASD** on your own resting-state EEG, with no programming required.
For each autistic subject it reports where the subject sits on the **excitation/inhibition (E/I) axis**
relative to a TD reference, and the **receptor modulation predicted to move that subject toward TD**.

**How to use it:** run each cell in order (click the play button, or press *Shift+Enter*).
Cells are numbered and self-explanatory. You do not need to edit any code.

> Neuron ASD is a research and hypothesis-generation tool. Its outputs are model-based predictions
> relative to a reference, **not** validated clinical prescriptions.


## 1 · Set up (run once)
This installs the required libraries, loads Neuron ASD, and prepares its response model.

**The first run takes a few minutes** because the response model has to be simulated once.
It is then saved, so if you come back to this notebook later it starts immediately.
All technical messages are hidden so the output stays clean.

In [ ]:
#@title Run to install and load Neuron ASD { display-mode: "form" }
import os, sys, subprocess, warnings, logging
os.environ['TF_CPP_MIN_LOG_LEVEL']='3'; os.environ['TF_ENABLE_ONEDNN_OPTS']='0'
os.environ['PYTHONWARNINGS']='ignore'
warnings.filterwarnings('ignore')

print('Installing Neuron ASD and its dependencies (this runs once)...')
_pkgs = ['mne', 'fooof', 'scikit-learn', 'tensorflow-cpu']
for _p in _pkgs:
    subprocess.run([sys.executable,'-m','pip','install','-q',_p],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# fetch the Neuron ASD package from the repository if not already present
if not os.path.exists('neuron_asd'):
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'git+https://github.com/arianadelg/neuron-asd.git'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    from neuron_asd import app
except Exception:
    # fallback: use local files if the notebook sits next to the package
    sys.path.insert(0, '.')
    from neuron_asd import app


# Build the response model once, now, so later steps return immediately.
app.prepare()

print('\nNeuron ASD is ready. You can move on to Step 2.')


## 2 · Provide the typically-developing (TD) reference
Neuron ASD compares each autistic subject against a **reference built from real TD recordings**.

**What to upload:** a single `.zip` file containing the resting-state EEG recordings of your TD group
(one file per subject). Supported formats include EEGLAB `.set`, EDF `.edf`, BioSemi `.bdf`,
BrainVision `.vhdr`, and FIF `.fif`.

**How many?** The reference becomes reliable at about **20 recordings**. With fewer, Neuron ASD still
runs but will warn you that some subjects may be misclassified. The TD group should come from the
**same cohort** as your subjects when possible; references from other datasets are not interchangeable.


In [ ]:
#@title Upload your TD reference (.zip) and build the reference
from google.colab import files
print('Select the .zip file with your TD recordings...')
_up = files.upload()
_td_zip = list(_up.keys())[0]

print('\nBuilding the TD reference (this may take a minute)...')
reference = app.build_reference(_td_zip)
app.reference_summary(reference)


## 3 · Analyze one subject
Upload a single autistic subject's resting-state EEG recording. Neuron ASD will show the E/I
placement and the recommended modulation, with a figure of the predicted effect toward TD.

In [ ]:
#@title Upload one subject and analyze
from google.colab import files
print('Select one EEG file for the subject...')
_up = files.upload()
_subj = list(_up.keys())[0]

result = app.analyze_subject(_subj, reference)
app.show(result)


## 4 · Analyze a whole group (optional)
To analyze many subjects at once, upload a `.zip` of their recordings. Neuron ASD returns a table
with one row per subject, which you can download as a spreadsheet (CSV).

In [ ]:
#@title Upload a group (.zip) and build the results table
from google.colab import files
print('Select the .zip file with your subject recordings...')
_up = files.upload()
_asd_zip = list(_up.keys())[0]

table = app.analyze_folder(_asd_zip, reference)
from IPython.display import display
display(table)

table.to_csv('neuron_asd_results.csv', index=False)
files.download('neuron_asd_results.csv')


## 5 · Just want to see it work? Try the built-in example
If you don't have data at hand, this cell downloads a tiny synthetic example bundled with the
repository and runs the full analysis, so you can see the expected output.

In [ ]:
#@title Run the built-in synthetic example
import os, urllib.request, zipfile
_base = 'https://raw.githubusercontent.com/arianadelg/neuron-asd/main/examples/'
for _f in ['example_td.zip', 'example_asd.zip']:
    if not os.path.exists(_f):
        try:
            urllib.request.urlretrieve(_base + _f, _f)
        except Exception as e:
            print('Could not download the example bundle:', e)

if os.path.exists('example_td.zip'):
    ref_demo = app.build_reference('example_td.zip')
    app.reference_summary(ref_demo)
    print()
    table_demo = app.analyze_folder('example_asd.zip', ref_demo)
    from IPython.display import display; display(table_demo)
else:
    print('Example not available offline; upload your own data in Steps 2-4 instead.')


---
### About the outputs
- **E/I placement** — where the subject sits on the excitation/inhibition axis (via the aperiodic
  1/f exponent) relative to the TD reference.
- **Recommended move** — the receptor agonist/inhibitor modulation predicted to move the subject's
  band profile toward TD, with a confidence (how consistently it was selected).
- **Predicted gain** — how far, in decibels and as a percentage, the recommended modulation is
  predicted to reduce the distance to TD.

For the methods, reliability conditions, and full description, see the accompanying paper and the
`USER_GUIDE.md` in the repository.
